# Features Analysis — CircuitNet-N28

Regex-parses filenames into a metadata table, builds per-sample statistical
descriptors of the 9 RouteNet input channels, tests which design knobs
actually drive separable feature-space structure, and validates several
federated-learning partitioning strategies with distributional metrics.

The pipeline is **evidence-based**: cluster / factor decisions are gated by
pre-committed thresholds declared in the imports cell (`ARI_MIN`,
`LR_ACC_MIN`, `MMD_P`, …), so partitioning axes get picked because they
pass the criteria — not because they win a post-hoc KS bake-off.

## Structure

| Section | Content |
|---------|---------|
| A  | Design parameter distributions (metadata parsed from filenames) |
| A2 | Phase 1 — DRC label sanity: per-sample violation scalars vs design knobs |
| B  | Per-channel statistical descriptors — extended block (mean, std, p50/p90/p99, Gini, nonzero fraction, spatial entropy, Moran's I) |
| C  | Per-channel distributions and inter-channel correlation |
| D  | Feature–metadata association (Pearson r, ANOVA η², R²) |
| E  | Whitened PCA of the descriptor block |
| E2 | Phase 2 — k-means + silhouette/DB/gap agreement, HDBSCAN-on-UMAP, ARI/NMI + logistic-regression encoding + PERMANOVA |
| F  | Partition-strategy validation (IID, per-pixel Noise, Synthetic grid, design-held-out, PPA persona, flow-recipe, Dirichlet over cluster labels) — scored with KS, Wasserstein, JS on labels, MMD on descriptors |
| G  | Summary — every recommendation gated by the pre-committed thresholds |

## Feature channels (RouteNet input order)

| # | Channel name |
|---|--------------|
| 0 | macro_region |
| 1 | cell_density |
| 2 | RUDY_long |
| 3 | RUDY_short |
| 4 | RUDY_pin_long |
| 5 | congestion_eGR_H |
| 6 | congestion_eGR_V |
| 7 | congestion_GR_H |
| 8 | congestion_GR_V |


In [ ]:
import os
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as mcm

from scipy import stats as scipy_stats
from scipy.stats import wasserstein_distance, ks_2samp, f_oneway
from scipy.spatial.distance import jensenshannon

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import (
    silhouette_score, davies_bouldin_score,
    adjusted_rand_score, normalized_mutual_info_score,
    roc_auc_score, accuracy_score,
)
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold

warnings.filterwarnings('ignore')
plt.rcParams.update({
    'figure.figsize': (14, 5),
    'axes.titlesize': 11,
    'axes.grid': True,
    'grid.color': 'white',
    'grid.linewidth': 0.8,
    'axes.facecolor': '#f5f5f5',
    'figure.facecolor': 'white',
})

sys.path.insert(0, os.path.abspath('.'))
from partitioning import (
    IIDPartitioner,
    NoiseFeaturePartitioner,
    SyntheticFeaturePartitioner,
    DirichletLabelPartitioner,
    LabelTierAssigner,
)

# ---------------------------------------------------------------------------
# Optional dependencies: HDBSCAN + UMAP are only used in Section E2 as a
# robustness cross-check on the k-means clustering. If they are not
# installed, the notebook still runs — E2 just skips that panel.
# ---------------------------------------------------------------------------
try:
    import umap  # type: ignore
    HAS_UMAP = True
except ImportError:
    HAS_UMAP = False
try:
    import hdbscan  # type: ignore
    HAS_HDBSCAN = True
except ImportError:
    HAS_HDBSCAN = False

# ---------------------------------------------------------------------------
# Data paths
# ---------------------------------------------------------------------------
FEATURE_DIR = '../drc_prediction/training_set/DRC/feature/'
LABEL_DIR   = '../drc_prediction/training_set/DRC/label/'   # DRC violation label maps

# ---------------------------------------------------------------------------
# Channel definitions (RouteNet input order)
# ---------------------------------------------------------------------------
N_CHANNELS = 9
CHANNEL_NAMES = [
    'macro_region', 'cell_density',
    'RUDY_long', 'RUDY_short', 'RUDY_pin_long',
    'congestion_eGR_H', 'congestion_eGR_V',
    'congestion_GR_H',  'congestion_GR_V',
]

# Channels whose per-pixel distribution is heavily right-skewed / sparse.
# For these we apply log1p before computing percentile / Gini-family stats,
# so a handful of extreme pixels does not dominate the descriptor.
SKEWED_CHANNELS = {
    'RUDY_long', 'RUDY_short', 'RUDY_pin_long',
    'congestion_eGR_H', 'congestion_eGR_V',
    'congestion_GR_H',  'congestion_GR_V',
}

# ---------------------------------------------------------------------------
# Pre-committed decision thresholds (CLAUDE.md § methodological decision 7).
# Fix these BEFORE running the analysis; Section G reads them back verbatim.
# ---------------------------------------------------------------------------
ARI_MIN           = 0.30   # A metadata factor is a valid partition axis if
                           # ARI(kmeans_labels, factor) >= ARI_MIN AND
LR_ACC_MIN        = 0.70   # logistic regression predicting the factor from
                           # PCA scores achieves accuracy >= LR_ACC_MIN.
DOMAIN_AUC_MAX    = 0.80   # (Cross-dataset shift check — used in the N14/N45
                           # comparison notebook, kept here for reference.)
MMD_P             = 0.01   # MMD permutation-test p-value below which we call
                           # a cross-partition feature shift significant.
NOISE_KS_FLOOR    = 0.05   # Real-noise level considered "meaningful" once its
                           # KS on channel-mean marginals exceeds this floor
                           # AND is >= 2x the IID KS baseline.
KMEANS_K_RANGE    = range(2, 16)  # k in [2, 15] for Phase 2 clustering.

# ---------------------------------------------------------------------------
# DRC label thresholds — same values as label_analysis.ipynb, kept here so
# Section A2 can run without a cross-notebook dependency.
# ---------------------------------------------------------------------------
VIOLATION_THRESHOLD = 0.1                     # per-pixel binarisation
LABEL_TIER_BOUNDARIES = [0.0, 0.02, 0.08, 0.20, 1.01]  # {clean, low, med, high}

print('Imports OK.')
print(f'  UMAP available:    {HAS_UMAP}')
print(f'  HDBSCAN available: {HAS_HDBSCAN}')


## Filename Parsing and Metadata Loading

In [ ]:
def parse_sample_name(filename: str) -> dict:
    """Parse a CircuitNet-N28 filename into its design-space components.

    Anchors from the right: the last 5 tokens always carry the prefixes
    c/u/m/p/f; the token at position -6 is macro_count; everything before
    that (after stripping any leading numeric id) is joined as the design name.

    Handles any number of hyphen-separated name parts, including:
      RISCY-a-1-c2-u0.7-m1-p1-f0.npy       -> design=RISCY-a
      1-RISCY-a-1-c2-u0.7-m1-p1-f0.npy     -> design=RISCY-a  (prefixed)
      zero-riscy-b-1-c2-u0.7-m1-p1-f0.npy  -> design=zero-riscy-b
    """
    basename = filename.replace('.npy', '')
    parts = basename.split('-')
    if parts[0].isdigit():
        parts = parts[1:]
    if len(parts) < 7:
        raise ValueError(f'Cannot parse (too few tokens): {filename}')
    for expected, token in zip(['c', 'u', 'm', 'p', 'f'], parts[-5:]):
        if not token.startswith(expected):
            raise ValueError(
                f'Cannot parse {filename}: expected prefix "{expected}", got "{token}"'
            )
    clock_str, util_str, macro_placement_raw, power_mesh_raw, filler_raw = parts[-5:]
    macro_count = parts[-6]
    design_name = '-'.join(parts[:-6])
    if not design_name:
        raise ValueError(f'Cannot parse {filename}: empty design name')
    return {
        'design_name':      design_name,
        'macro_count':      macro_count,
        'clock_ns':         float(clock_str[1:]),
        'utilization':      float(util_str[1:]),
        'macro_placement':  macro_placement_raw[1:],
        'power_mesh':       power_mesh_raw[1:],
        'filler_insertion': filler_raw[1:],
        'filename':         filename,
    }


feature_exists = os.path.isdir(FEATURE_DIR)
files = sorted(f for f in os.listdir(FEATURE_DIR) if f.endswith('.npy'))
print(f'Feature directory found: {len(files)} .npy files.')
records = []
for fname in files:
    try:
        records.append(parse_sample_name(fname))
    except Exception as e:
        print(f'  Skip {fname}: {e}')
df_meta = pd.DataFrame(records)

print(f'Samples: {len(df_meta)}')
print(df_meta.head())
print(df_meta.dtypes)

# macro_placement occasionally carries out-of-range tokens; coerce to 1 then int.
df_meta.loc[df_meta['macro_placement'].isin(['1', '2', '3', '4']) == False,
            'macro_placement'] = '1'
df_meta['macro_placement'] = df_meta['macro_placement'].astype(int)

# --------------------------------------------------------------------------
# Convenience: sorted list of unique design names — used by every downstream
# section that groups / iterates over designs (fixes a NameError that was
# hidden in the v1 notebook when a helper cell was deleted).
# --------------------------------------------------------------------------
designs_uniq = sorted(df_meta['design_name'].unique().tolist())
print(f'Unique designs ({len(designs_uniq)}): {designs_uniq}')


---
## Section A — Design Parameter Distributions

Understand the design space before looking at feature values. These parameters
directly control routing difficulty and are the natural axes for feature-based partitioning.

In [ ]:
CAT_COLS = ['design_name','macro_count','macro_placement','power_mesh','filler_insertion']
NUM_COLS = ['utilization','clock_ns']

print('=== Categorical parameter distributions ===')
for col in CAT_COLS:
    vc = df_meta[col].value_counts()
    balance = vc.min() / vc.max()
    print(f'\n{col}  (unique={vc.shape[0]}, balance={balance:.3f}):')
    print(vc.to_string())

print('\n=== Numerical parameter distributions ===')
for col in NUM_COLS:
    print(f'\n{col}:')
    print(df_meta[col].describe().to_string())
    print('  Unique values:', sorted(df_meta[col].unique()))


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(24, 10))
fig.suptitle('Section A — Design Parameter Distributions', fontsize=14, fontweight='bold')

cmap10 = mcm.get_cmap('tab10')

for ax, col in zip(axes[0], CAT_COLS):
    vc = df_meta[col].value_counts().sort_index()
    colors = [cmap10(i % 10) for i in range(len(vc))]
    bars = ax.bar(range(len(vc)), vc.values, color=colors, alpha=0.85, edgecolor='white')
    ax.set_xticks(range(len(vc)))
    ax.set_xticklabels(vc.index, rotation=35, ha='right', fontsize=8)
    ax.set_title(col)
    ax.set_ylabel('Count')
    for bar, cnt in zip(bars, vc.values):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+5,
                str(cnt), ha='center', fontsize=7)

# Utilization histogram
ax = axes[0][3]
ax.hist(df_meta['utilization'], bins=30, color=cmap10(5), alpha=0.85, edgecolor='white')
ax.set_title('utilization')
ax.set_xlabel('utilization ratio')
ax.set_ylabel('Count')

# Clock histogram
ax = axes[1][0]
ax.hist(df_meta['clock_ns'], bins=20, color=cmap10(6), alpha=0.85, edgecolor='white')
ax.set_title('clock_ns')
ax.set_xlabel('Clock period (ns)')
ax.set_ylabel('Count')

# design x macro_placement
ax = axes[1][1]
pivot2 = df_meta.groupby(['design_name', 'macro_placement']).size().unstack(fill_value=0)
mat2 = pivot2.values.astype(float)
im2 = ax.imshow(mat2, aspect='auto', cmap='Purples')
ax.set_yticks(range(len(pivot2.index)))
ax.set_yticklabels(pivot2.index, fontsize=8)
ax.set_xticks(range(len(pivot2.columns)))
ax.set_xticklabels([str(c) for c in pivot2.columns], rotation=30, ha='right', fontsize=7)
ax.set_title('Sample count: design × macro_placement')
plt.colorbar(im2, ax=ax, fraction=0.04)

# Design × utilization sample counts heatmap
ax = axes[1][2]
util_bins = pd.cut(df_meta['utilization'], bins=5)
pivot = df_meta.groupby(['design_name', util_bins]).size().unstack(fill_value=0)
mat = pivot.values.astype(float)
im = ax.imshow(mat, aspect='auto', cmap='Blues')
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index, fontsize=8)
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels([str(c) for c in pivot.columns], rotation=30, ha='right', fontsize=7)
ax.set_title('Sample count: design × utilization bin')
plt.colorbar(im, ax=ax, fraction=0.04)

# Design × clock sample counts heatmap
ax = axes[1][3]
clock_bins = pd.cut(df_meta['clock_ns'], bins=5)
pivot2 = df_meta.groupby(['design_name', clock_bins]).size().unstack(fill_value=0)
mat2 = pivot2.values.astype(float)
im2 = ax.imshow(mat2, aspect='auto', cmap='Purples')
ax.set_yticks(range(len(pivot2.index)))
ax.set_yticklabels(pivot2.index, fontsize=8)
ax.set_xticks(range(len(pivot2.columns)))
ax.set_xticklabels([str(c) for c in pivot2.columns], rotation=30, ha='right', fontsize=7)
ax.set_title('Sample count: design × clock_ns bin')
plt.colorbar(im2, ax=ax, fraction=0.04)

plt.tight_layout()
plt.show()

# Design space coverage
all_cols = CAT_COLS + ['clock_ns', 'utilization']
unique_combos = df_meta[all_cols].drop_duplicates()
print(f'Unique parameter combinations: {len(unique_combos)} / {len(df_meta)} samples')

---
## Section A2 — Phase 1: DRC label sanity

CLAUDE.md Phase 1 explicitly requires joining per-sample DRC scalars
(violation count, violation rate, tier) onto the metadata table so we can
answer basic sanity questions **before** doing any feature analysis:

- Do harder routing knobs (higher utilization, tighter clock, more macros)
  actually produce more DRC violations?
- What fraction of samples are in each severity tier — is the label heavily
  imbalanced? (This dictates whether Dirichlet-over-labels is a meaningful
  partitioning strategy at all.)

This section loads the DRC label maps once, reduces each to a small handful
of scalars using the same `LabelTierAssigner` used by `label_analysis.ipynb`,
and produces the three "DRC vs knob" plots that CLAUDE.md prescribes.

The rest of the notebook does not consume `df_label` as an input feature —
labels only reappear in Section F to score partitions on JS divergence of
their tier distributions and to define the Dirichlet-label partitioner.


In [ ]:
# -----------------------------------------------------------------
# A2.1 — Load DRC label maps and reduce each to three per-sample scalars.
#
# We do NOT train any model on labels here; this block only computes:
#   - violation_count   : how many pixels exceed VIOLATION_THRESHOLD
#   - violation_rate    : same, normalised by the number of pixels
#   - tier              : {0,1,2,3} = {clean, low, med, high} bucket used
#                         later by the Dirichlet partitioner
# The rate/tier are what CLAUDE.md Phase 1 calls the "per-sample DRC scalar"
# needed to join labels onto metadata and to score partitions on JS
# divergence of label distributions.
#
# Cost: one np.load per label map. For 10k samples this is a few minutes
# once; the result is cached in-memory in `df_label` for the rest of the
# notebook.
# -----------------------------------------------------------------
label_exists = os.path.isdir(LABEL_DIR)
print(f'Label directory found: {label_exists} -> {LABEL_DIR}')

df_label = pd.DataFrame(columns=['filename', 'violation_count',
                                 'violation_rate', 'tier'])

if label_exists:
    tier_assigner = LabelTierAssigner(
        threshold=VIOLATION_THRESHOLD,
        boundaries=LABEL_TIER_BOUNDARIES,
    )
    label_rows = []
    for fname in df_meta['filename']:
        lpath = os.path.join(LABEL_DIR, fname)
        if not os.path.isfile(lpath):
            continue
        try:
            arr = np.load(lpath)
            stats = tier_assigner.label_stats(arr)
            label_rows.append({
                'filename':        fname,
                'violation_count': stats['violation_count'],
                'violation_rate':  stats['violation_rate'],
                'tier':            stats['tier'],
            })
        except Exception as e:
            # Skip broken files silently; a print-per-file would flood the
            # output for a 10k-sample loop.
            continue
    df_label = pd.DataFrame(label_rows)
    print(f'DRC labels processed: {len(df_label)} / {len(df_meta)} samples.')

    if len(df_label) > 0:
        print('\nDRC scalar summary:')
        print(df_label[['violation_count', 'violation_rate']]
              .describe().round(4).to_string())
        print('\nSeverity-tier distribution:')
        tier_counts = df_label['tier'].value_counts().sort_index()
        for t, c in tier_counts.items():
            name = LabelTierAssigner.TIER_NAMES.get(t, '?')
            print(f'  tier {t} ({name:<6s}): {c:>5d}  ({c/len(df_label):.1%})')
else:
    print('Skipping DRC label load — LABEL_DIR does not exist.')


In [ ]:
# -----------------------------------------------------------------
# A2.2 — Sanity plots: DRC severity vs each design knob.
#
# Interpretation guide
#   - If DRC violation-rate rises with utilization, clocks tightening,
#     or macro count, then routing difficulty is genuinely tracking those
#     knobs — which validates using (utilization, clock_ns, macro_count)
#     as partitioning axes for feature-driven Non-IID splits.
#   - If a knob shows no monotonic effect on DRC (e.g. filler insertion),
#     it is a poor persona axis and should not be used for partitioning.
# -----------------------------------------------------------------
if 'df_label' in globals() and len(df_label) > 0:
    df_ann = df_meta.merge(df_label, on='filename', how='inner')

    fig, axes = plt.subplots(1, 3, figsize=(21, 5))
    fig.suptitle('Section A2 — DRC severity vs design knobs',
                 fontsize=13, fontweight='bold')

    # (a) Utilization vs violation rate — expect positive slope.
    ax = axes[0]
    for u in sorted(df_ann['utilization'].unique()):
        sub = df_ann[df_ann['utilization'] == u]['violation_rate'].values
        ax.scatter([u] * len(sub), sub, s=5, alpha=0.2, color='steelblue')
    means = df_ann.groupby('utilization')['violation_rate'].mean()
    ax.plot(means.index, means.values, 'r-o', linewidth=2, label='mean')
    ax.set_xlabel('utilization')
    ax.set_ylabel('DRC violation rate')
    ax.set_title('violation rate vs utilization')
    ax.legend(fontsize=9)

    # (b) Clock period vs violation rate — expect NEGATIVE slope
    # (shorter clock = tighter timing = harder routing).
    ax = axes[1]
    for c in sorted(df_ann['clock_ns'].unique()):
        sub = df_ann[df_ann['clock_ns'] == c]['violation_rate'].values
        ax.scatter([c] * len(sub), sub, s=5, alpha=0.2, color='orange')
    means_c = df_ann.groupby('clock_ns')['violation_rate'].mean()
    ax.plot(means_c.index, means_c.values, 'r-o', linewidth=2, label='mean')
    ax.set_xlabel('clock_ns (ns per cycle)')
    ax.set_ylabel('DRC violation rate')
    ax.set_title('violation rate vs clock period')
    ax.legend(fontsize=9)

    # (c) Macro count vs violation rate — expect roughly monotonic.
    ax = axes[2]
    for m in sorted(df_ann['macro_count'].unique()):
        sub = df_ann[df_ann['macro_count'] == m]['violation_rate'].values
        ax.scatter([m] * len(sub), sub, s=5, alpha=0.2, color='purple')
    means_m = df_ann.groupby('macro_count')['violation_rate'].mean()
    ax.plot(means_m.index, means_m.values, 'r-o', linewidth=2, label='mean')
    ax.set_xlabel('macro_count')
    ax.set_ylabel('DRC violation rate')
    ax.set_title('violation rate vs #macros')
    ax.legend(fontsize=9)

    plt.tight_layout()
    plt.show()

    # Numeric summary — quick monotonicity check with Spearman rho.
    for knob in ['utilization', 'clock_ns', 'macro_count']:
        rho, p = scipy_stats.spearmanr(df_ann[knob].astype(float),
                                       df_ann['violation_rate'])
        marker = '***' if p < 1e-3 else ('*' if p < 0.05 else 'n.s.')
        print(f'  Spearman rho(violation_rate ~ {knob:12s}) = {rho:+.3f}  ({marker})')
else:
    print('df_label not available -> skipping DRC-vs-knobs sanity plots.')


---
## Section B — Feature Channel Statistics (extended descriptor block)

For each sample and each of the 9 feature channels we build an
9-dimensional descriptor:

| Stat            | What it captures                                                                 |
|-----------------|-----------------------------------------------------------------------------------|
| `mean`          | Overall intensity of the channel across the tile map                              |
| `std`           | Spatial variation — flat maps vs textured maps                                    |
| `p50/p90/p99`   | Robust quantiles — insensitive to a handful of outliers, unlike `max`             |
| `gini`          | Concentration of the intensity mass — 0 = uniform, 1 = one hot pixel              |
| `nonzero_frac`  | Fraction of tiles that are active — sparsity of routing demand                    |
| `spatial_ent`   | 2-D Shannon entropy over binned intensities — high = uniform, low = concentrated  |
| `moran_i`       | Rook-neighbour Moran's I — spatial autocorrelation, high = clustered hotspots     |

That is **9 stats x 9 channels = 81 descriptors per sample**, giving the PCA
and clustering steps a much richer view of the maps than v1's 4-stat block
(which included a degenerate `max = 1.0` on 6/9 channels).

The 7 congestion / RUDY channels are `log1p`-transformed *before* percentile
and Gini stats are computed, because their raw pixel values are heavily
right-skewed. Without the log a few extreme pixels would dominate every
percentile and pull the descriptor onto essentially the same axis for all
samples.

We downsample each map to 64x64 for Moran's I and spatial entropy — full
256x256 would be exact but ~16x more expensive with no meaningful accuracy
gain for a scalar descriptor. Mean / std / percentiles are still computed
on the full-resolution map.


In [ ]:
# --------------------------------------------------------------------
# The nine per-channel scalar statistics we compute for every sample.
# Order matters: `mean` MUST stay first because downstream code refers
# to `f'{ch}_mean'` (e.g. Section D's ANOVA, Section F's pairwise KS).
# --------------------------------------------------------------------
STATS = ['mean', 'std', 'p50', 'p90', 'p99',
         'gini', 'nonzero_frac', 'spatial_ent', 'moran_i']
FEAT_COLS = [f'{ch}_{stat}' for ch in CHANNEL_NAMES for stat in STATS]
mean_cols = [f'{ch}_mean' for ch in CHANNEL_NAMES]   # convenience alias


def _gini(x: np.ndarray) -> float:
    """Gini coefficient of a non-negative 1-D array.

    0 = perfectly uniform, 1 = one pixel carries everything.
    Cheaper than the O(n^2) formula: sort once and use the mean-based form.
    """
    v = np.abs(x).ravel()
    if v.size == 0:
        return 0.0
    v = np.sort(v)
    n = v.size
    cum = np.cumsum(v)
    total = cum[-1]
    if total <= 0:
        return 0.0
    # 2 * area between the Lorenz curve and the 45-degree line.
    return float((2.0 * np.sum((np.arange(1, n + 1)) * v)) / (n * total) - (n + 1) / n)


def _spatial_entropy(m: np.ndarray, bins: int = 16) -> float:
    """Shannon entropy (nats) of the pixel-intensity histogram of a 2-D map.

    High -> intensities are spread uniformly, no dominant value.
    Low  -> most pixels sit in a single bin.
    """
    finite = m[np.isfinite(m)]
    if finite.size == 0:
        return 0.0
    hist, _ = np.histogram(finite, bins=bins)
    p = hist.astype(np.float64)
    p = p[p > 0]
    if p.size == 0:
        return 0.0
    p /= p.sum()
    return float(-(p * np.log(p)).sum())


def _moran_i(m: np.ndarray) -> float:
    """Rook-neighbour Moran's I for a 2-D map.

    Positive -> hot pixels cluster spatially (typical for congestion hotspots).
    Zero     -> random pattern.
    Negative -> checker-board pattern (rare in this dataset).
    Downsampled first so cost is O(H*W) with a small H,W.
    """
    if m.size == 0:
        return 0.0
    mean = m.mean()
    dev = m - mean
    denom = float((dev ** 2).sum())
    if denom == 0.0:
        return 0.0
    # Sum of dev[i,j]*dev[neighbour] over the 4 rook neighbours.
    num = float((dev[:-1, :] * dev[1:, :]).sum() +
                (dev[:, :-1] * dev[:, 1:]).sum())
    W = 2 * (m.shape[0] - 1) * m.shape[1]      # horizontal + vertical edges
    W += 2 * m.shape[0] * (m.shape[1] - 1)
    if W == 0:
        return 0.0
    return float(m.size * num / (W * denom))


def _downsample_2d(m: np.ndarray, target: int = 64) -> np.ndarray:
    """Area-averaged downsample of a 2-D map to (target, target).

    Uses block-mean over integer factors -- exactly the safe reducer
    CLAUDE.md recommends for cross-scale comparisons.
    """
    h, w = m.shape
    if h <= target and w <= target:
        return m.astype(np.float32, copy=False)
    fh = max(1, h // target)
    fw = max(1, w // target)
    trim_h = (h // fh) * fh
    trim_w = (w // fw) * fw
    return (m[:trim_h, :trim_w]
              .reshape(trim_h // fh, fh, trim_w // fw, fw)
              .mean(axis=(1, 3))
              .astype(np.float32, copy=False))


def extract_channel_stats(arr: np.ndarray) -> np.ndarray:
    """Compute the 9-stat descriptor for every channel of one sample.

    Returns a (81,) vector ordered by CHANNEL_NAMES x STATS.
    """
    # Reorder to (C, H, W) so channel iteration is uniform.
    if arr.ndim == 3 and arr.shape[-1] == N_CHANNELS:
        arr = np.transpose(arr, (2, 0, 1))
    elif arr.ndim == 3 and arr.shape[0] == N_CHANNELS:
        pass
    else:
        raise ValueError(f'Unexpected shape {arr.shape} for {N_CHANNELS}-channel feature')

    row = []
    for c, ch_name in enumerate(CHANNEL_NAMES):
        m = arr[c].astype(np.float32, copy=False)
        # For skewed channels compress the dynamic range BEFORE computing
        # percentile / concentration stats, but keep the raw mean/std/nonzero
        # so those numbers remain physically interpretable.
        m_stat = np.log1p(np.clip(m, 0, None)) if ch_name in SKEWED_CHANNELS else m
        flat_raw  = m.ravel()
        flat_stat = m_stat.ravel()

        mean = float(flat_raw.mean())
        std  = float(flat_raw.std())
        p50  = float(np.percentile(flat_stat, 50))
        p90  = float(np.percentile(flat_stat, 90))
        p99  = float(np.percentile(flat_stat, 99))
        gini = _gini(flat_stat)
        nz   = float(np.mean(flat_raw > 1e-8))
        m_ds = _downsample_2d(m_stat, target=64)
        s_ent = _spatial_entropy(m_ds, bins=16)
        mor   = _moran_i(m_ds)

        row.extend([mean, std, p50, p90, p99, gini, nz, s_ent, mor])

    return np.array(row, dtype=np.float32)


print(f'Extracting extended descriptor block ({len(STATS)} stats x '
      f'{N_CHANNELS} channels = {len(FEAT_COLS)} dims) '
      f'from {len(df_meta)} feature files...')
stat_rows = []
failed = 0
for fname in df_meta['filename']:
    fpath = os.path.join(FEATURE_DIR, fname)
    try:
        arr = np.load(fpath)
        stat_rows.append(extract_channel_stats(arr))
    except Exception:
        stat_rows.append(np.full(len(FEAT_COLS), np.nan, dtype=np.float32))
        failed += 1

df_stats = pd.DataFrame(stat_rows, columns=FEAT_COLS)
df = pd.concat([df_meta.reset_index(drop=True), df_stats], axis=1)
df = df.dropna(subset=FEAT_COLS).reset_index(drop=True)
print(f'Done. {len(df)} / {len(df_meta)} samples loaded successfully ({failed} failed).')

# One-line summary the eye can scan — full 81-column describe would be huge.
print(f'\nFeature statistics shape: {df[FEAT_COLS].shape}')
print(df[mean_cols].describe().round(4).to_string())


---
## Section C — Per-Channel Distribution and Inter-Channel Correlation

Understand the marginal distribution of each channel's mean and std, and identify
redundant channels (high correlation) vs independent channels.

In [ ]:
# Distribution of channel *means* across samples
mean_cols = [f'{ch}_mean' for ch in CHANNEL_NAMES]

fig, axes = plt.subplots(3, 3, figsize=(18, 12))
fig.suptitle('Section C — Per-Channel Mean Distribution (across 10 k samples)',
             fontsize=13, fontweight='bold')

palette = mcm.get_cmap('tab10')
for ax, col, i in zip(axes.ravel(), mean_cols, range(N_CHANNELS)):
    vals = df[col].values
    ax.hist(vals, bins=60, color=palette(i), alpha=0.8, edgecolor='white')
    ax.axvline(vals.mean(), color='black', linestyle='--', linewidth=1.2,
               label=f'mean={vals.mean():.3f}')
    ax.axvline(np.median(vals), color='red', linestyle=':', linewidth=1.0,
               label=f'median={np.median(vals):.3f}')
    ax.set_title(CHANNEL_NAMES[i])
    ax.set_xlabel('channel mean')
    ax.set_ylabel('count')
    ax.legend(fontsize=7)

plt.tight_layout()
plt.show()

In [ ]:
# Pearson correlation matrix of the 9 channel means
corr = df[mean_cols].corr()

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle('Section C — Inter-Channel Correlation', fontsize=13, fontweight='bold')

ax = axes[0]
mat = corr.values
im = ax.imshow(mat, cmap='RdBu_r', vmin=-1, vmax=1)
for r in range(N_CHANNELS):
    for c in range(N_CHANNELS):
        ax.text(c, r, f'{mat[r,c]:.2f}', ha='center', va='center',
                fontsize=7, color='white' if abs(mat[r,c]) > 0.6 else 'black')
ax.set_xticks(range(N_CHANNELS))
ax.set_xticklabels(CHANNEL_NAMES, rotation=45, ha='right', fontsize=8)
ax.set_yticks(range(N_CHANNELS))
ax.set_yticklabels(CHANNEL_NAMES, fontsize=8)
ax.set_title('Pearson correlation of channel means')
plt.colorbar(im, ax=ax, fraction=0.04)

# Variance per channel (which channels have most spread across samples?)
ax = axes[1]
channel_var = df[mean_cols].var().values
bars = ax.bar(range(N_CHANNELS), channel_var,
              color=[palette(i) for i in range(N_CHANNELS)], alpha=0.85, edgecolor='white')
ax.set_xticks(range(N_CHANNELS))
ax.set_xticklabels(CHANNEL_NAMES, rotation=45, ha='right', fontsize=8)
ax.set_title('Variance of channel mean across samples\n(higher → more informative for partitioning)')
ax.set_ylabel('Variance')
for bar, v in zip(bars, channel_var):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.02,
            f'{v:.4f}', ha='center', fontsize=7, rotation=60)

plt.tight_layout()
plt.show()

# Print highly correlated pairs
print('Highly correlated channel pairs (|r| > 0.7):')
found = False
for i in range(N_CHANNELS):
    for j in range(i+1, N_CHANNELS):
        r = mat[i, j]
        if abs(r) > 0.7:
            print(f'  {CHANNEL_NAMES[i]} ↔ {CHANNEL_NAMES[j]}: r = {r:.3f}')
            found = True
if not found:
    print('  None found above |r| = 0.7')

---
## Section D — Feature–Metadata Correlation

Identify which design knobs (utilization, clock_ns, design_name) are the strongest
predictors of feature channel values.

This directly informs which metadata axes are valid for **Synthetic Feature Partitioning** (NIID-Bench strategy 4):
only axes that strongly predict features create genuine feature-distribution skew between parties.

In [ ]:
# Pearson r between each continuous metadata variable and each channel mean
meta_num = ['utilization', 'clock_ns']
print('Pearson r between design parameters and channel means:')
print(f'{"":20s}', end='')
for ch in CHANNEL_NAMES:
    print(f'{ch[:12]:>13s}', end='')
print()

corr_rows = {}
for param in meta_num:
    corr_rows[param] = []
    print(f'{param:20s}', end='')
    for col in mean_cols:
        r, _ = scipy_stats.pearsonr(df[param], df[col])
        corr_rows[param].append(r)
        print(f'{r:>13.3f}', end='')
    print()

# ANOVA F-statistic: does design_name significantly explain each channel?
print('\nOne-way ANOVA F (design_name → channel mean):')
print(f'{"":20s}', end='')
for ch in CHANNEL_NAMES:
    print(f'{ch[:12]:>13s}', end='')
print()

anova_f = []
anova_p = []
print(f'{"design_name (F)":20s}', end='')
for col in mean_cols:
    groups = [df.loc[df['design_name']==d, col].values for d in designs_uniq]
    stat, pval = f_oneway(*groups)
    anova_f.append(stat)
    anova_p.append(pval)
    print(f'{stat:>13.1f}', end='')
print()

print(f'{"design_name (p)":20s}', end='')
for p in anova_p:
    marker = '***' if p < 0.001 else ('*' if p < 0.05 else '  n.s.')
    print(f'{marker:>13s}', end='')
print()

# Eta-squared (effect size for ANOVA)
eta_sq = []
print(f'{"design_name (η²)":20s}', end='')
for col in mean_cols:
    groups = [df.loc[df['design_name']==d, col].values for d in designs_uniq]
    grand_mean = df[col].mean()
    ss_between = sum(len(g)*(g.mean()-grand_mean)**2 for g in groups)
    ss_total   = sum((df[col].values - grand_mean)**2)
    eta2 = ss_between / ss_total if ss_total > 0 else 0.0
    eta_sq.append(eta2)
    print(f'{eta2:>13.3f}', end='')
print()

print('\nInterpretation: η² > 0.14 → large effect, 0.06-0.14 → medium, < 0.06 → small')

In [ ]:
# Heatmap: Pearson r between design params + design_name (as eta^2) and channel means
fig, axes = plt.subplots(1, 2, figsize=(18, 5))
fig.suptitle('Section D — Feature–Metadata Correlation Strength',
             fontsize=13, fontweight='bold')

# Build combined correlation matrix
corr_matrix = np.array(
    [corr_rows['utilization'],
     corr_rows['clock_ns'],
     eta_sq]
)
row_labels = ['utilization (r)', 'clock_ns (r)', 'design_name (η²)']

ax = axes[0]
im = ax.imshow(corr_matrix, aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1)
for r in range(corr_matrix.shape[0]):
    for c in range(N_CHANNELS):
        val = corr_matrix[r, c]
        ax.text(c, r, f'{val:.2f}', ha='center', va='center',
                fontsize=8, color='white' if abs(val) > 0.6 else 'black')
ax.set_xticks(range(N_CHANNELS))
ax.set_xticklabels(CHANNEL_NAMES, rotation=45, ha='right', fontsize=8)
ax.set_yticks(range(3))
ax.set_yticklabels(row_labels, fontsize=9)
ax.set_title('Correlation / effect size (r for continuous, η² for design)')
plt.colorbar(im, ax=ax, fraction=0.03)

# Box plots: cell_density and congestion_GR_H per design
ax = axes[1]
key_channels = ['cell_density_mean', 'congestion_GR_H_mean']
offsets = np.linspace(-0.2, 0.2, len(key_channels))
design_idx = {d: i for i, d in enumerate(designs_uniq)}
colors_key = ['steelblue', 'darkorange']

for j, (col, color) in enumerate(zip(key_channels, colors_key)):
    positions = [design_idx[d] + offsets[j] for d in designs_uniq]
    data = [df.loc[df['design_name']==d, col].values for d in designs_uniq]
    bp = ax.boxplot(data, positions=positions, widths=0.3,
                    patch_artist=True, notch=False,
                    medianprops={'color':'black','linewidth':1.3},
                    flierprops={'marker':'.','markersize':2,'alpha':0.3})
    for patch in bp['boxes']:
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    bp['boxes'][0].set_label(col.replace('_mean',''))

ax.set_xticks(range(len(designs_uniq)))
ax.set_xticklabels(designs_uniq, rotation=30, ha='right', fontsize=8)
ax.set_title('cell_density & congestion_GR_H per design')
ax.set_ylabel('channel mean')
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Which channels are best predicted by utilization and clock_ns?
# Use linear regression R² for a clean summary
from sklearn.linear_model import LinearRegression

X_meta = df[['utilization', 'clock_ns']].values
r2_scores = {}
for col in mean_cols:
    y = df[col].values
    reg = LinearRegression().fit(X_meta, y)
    r2_scores[col] = reg.score(X_meta, y)

r2_series = pd.Series(r2_scores, name='R²')
print('R² of (utilization + clock_ns) → channel mean:')
print(r2_series.sort_values(ascending=False).round(4).to_string())

fig, ax = plt.subplots(figsize=(10, 4))
colors_r2 = ['#4caf50' if v > 0.3 else '#ff9800' if v > 0.1 else '#f44336'
              for v in r2_series.values]
bars = ax.bar(range(len(r2_series)), r2_series.values, color=colors_r2, alpha=0.85, edgecolor='white')
ax.set_xticks(range(len(r2_series)))
ax.set_xticklabels([c.replace('_mean','') for c in r2_series.index],
                   rotation=40, ha='right', fontsize=9)
ax.axhline(0.30, color='green',  linestyle='--', linewidth=1, label='R²=0.30 (good predictor)')
ax.axhline(0.10, color='orange', linestyle='--', linewidth=1, label='R²=0.10 (weak predictor)')
ax.set_title('R² of (utilization, clock_ns) → channel mean\n'
             'Green → strong: valid Synthetic partitioning axis')
ax.set_ylabel('R²')
ax.legend(fontsize=9)
for bar, v in zip(bars, r2_series.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.003,
            f'{v:.2f}', ha='center', fontsize=8)
plt.tight_layout()
plt.show()

---
## Section E — Whitened PCA of the descriptor block

We project the **81-dimensional descriptor vector** onto principal
components. Two design choices matter here:

1. **StandardScaler first.** Each stat has its own unit (mean is in
   [0,1], Moran's I in [-1,1], spatial_ent is unbounded). Without
   standardisation the PCA would just discover which stats have the largest
   raw variance.
2. **`whiten=True`.** Without whitening, PC1 captures the "overall design
   size" axis and dominates every Euclidean distance downstream — meaning
   k-means in E2 would degenerate into "sort by design size", which is
   uninformative. Whitening rescales each PC to unit variance so
   downstream clustering treats every direction on equal footing.
   (CLAUDE.md § methodological decision 5 makes this explicit.)

We fit enough components to reach at least **90 % cumulative variance** —
whichever k that turns out to be. That k also becomes the input
dimensionality for Section E2's k-means and logistic-regression tests.


In [ ]:
# --------------------------------------------------------------------
# Step 1: standardise every descriptor to zero mean, unit variance.
# --------------------------------------------------------------------
X_raw    = df[FEAT_COLS].values
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

# --------------------------------------------------------------------
# Step 2: probe the spectrum with a non-whitened full PCA to decide how
# many components we need to hit >= 90% cumulative variance.
# --------------------------------------------------------------------
probe = PCA(n_components=min(X_scaled.shape[1], X_scaled.shape[0] - 1))
probe.fit(X_scaled)
cum_probe = np.cumsum(probe.explained_variance_ratio_)
n_for_90  = int(np.searchsorted(cum_probe, 0.90)) + 1
print(f'PCs needed for >= 90% variance: {n_for_90} '
      f'(out of {len(probe.explained_variance_ratio_)} available)')

# --------------------------------------------------------------------
# Step 3: refit with whitening at that dimensionality. This is the
# representation every downstream section (E-scatter, E2, F-MMD) uses.
# --------------------------------------------------------------------
pca   = PCA(n_components=n_for_90, whiten=True)
X_pca = pca.fit_transform(X_scaled)

explained  = pca.explained_variance_ratio_
cumulative = np.cumsum(explained)

print('\nPCA explained variance ratio (whitened, top-10 shown):')
for i, (ev, cv) in enumerate(zip(explained[:10], cumulative[:10])):
    print(f'  PC{i+1:>2d}: {ev:.3f}  (cumulative: {cv:.3f})')

# --------------------------------------------------------------------
# Two-panel visual: scree curve on the left, top-12 PC1 loadings on the
# right (loadings tell us which descriptors drive the first axis; before
# whitening this used to be dominated by "design size").
# --------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Section E - PCA of Feature Statistics (whitened)',
             fontsize=13, fontweight='bold')

ax = axes[0]
ax.bar(range(1, len(explained)+1), explained,
       color='steelblue', alpha=0.8, edgecolor='white')
ax.plot(range(1, len(cumulative)+1), cumulative,
        'ro-', linewidth=1.5, markersize=5, label='Cumulative')
ax.axhline(0.90, color='red', linestyle='--', linewidth=1, label='90% threshold')
ax.set_xlabel('Principal Component')
ax.set_ylabel('Explained variance ratio')
ax.set_title(f'Scree plot ({n_for_90} PCs retained)')
ax.legend(fontsize=9)

ax = axes[1]
loadings_pc1 = pca.components_[0]
sorted_idx   = np.argsort(np.abs(loadings_pc1))[::-1][:12]
colors_load  = ['#4caf50' if v > 0 else '#f44336' for v in loadings_pc1[sorted_idx]]
ax.bar(range(12), loadings_pc1[sorted_idx],
       color=colors_load, alpha=0.85, edgecolor='white')
ax.set_xticks(range(12))
ax.set_xticklabels(
    [FEAT_COLS[i].replace('congestion_', 'cong_') for i in sorted_idx],
    rotation=45, ha='right', fontsize=8,
)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('Top-12 PC1 loadings (which descriptors drive PC1)')
ax.set_ylabel('Loading')
plt.tight_layout()
plt.show()


In [ ]:
# PCA scatter coloured by design_name, utilization, clock_ns
N_PLOT = min(4000, len(df))
idx_plot = np.random.default_rng(0).choice(len(df), N_PLOT, replace=False)
X2 = X_pca[idx_plot, :2]

fig, axes = plt.subplots(1, 3, figsize=(21, 6))
fig.suptitle('Section E — PCA Scatter (PC1 vs PC2)', fontsize=13, fontweight='bold')

# By design
ax = axes[0]
cmap_d = mcm.get_cmap('Set2', len(designs_uniq))
for i, d in enumerate(designs_uniq):
    mask = df.iloc[idx_plot]['design_name'].values == d
    ax.scatter(X2[mask, 0], X2[mask, 1], s=5, alpha=0.35,
               color=cmap_d(i), label=d)
ax.set_xlabel(f'PC1 ({explained[0]:.1%})')
ax.set_ylabel(f'PC2 ({explained[1]:.1%})')
ax.set_title('Coloured by design_name')
ax.legend(fontsize=7, markerscale=4)

# By utilization
ax = axes[1]
sc = ax.scatter(X2[:, 0], X2[:, 1], s=4, alpha=0.35,
                c=df.iloc[idx_plot]['utilization'].values, cmap='plasma')
plt.colorbar(sc, ax=ax, label='utilization')
ax.set_xlabel(f'PC1 ({explained[0]:.1%})')
ax.set_ylabel(f'PC2 ({explained[1]:.1%})')
ax.set_title('Coloured by utilization')

# By clock_ns
ax = axes[2]
sc = ax.scatter(X2[:, 0], X2[:, 1], s=4, alpha=0.35,
                c=df.iloc[idx_plot]['clock_ns'].values, cmap='viridis')
plt.colorbar(sc, ax=ax, label='clock_ns')
ax.set_xlabel(f'PC1 ({explained[0]:.1%})')
ax.set_ylabel(f'PC2 ({explained[1]:.1%})')
ax.set_title('Coloured by clock_ns')

plt.tight_layout()
plt.show()

# Quantify: how much PC1 variance is explained by utilization alone?
r_util_pc1, _ = scipy_stats.pearsonr(df['utilization'], X_pca[:, 0])
r_clock_pc1, _ = scipy_stats.pearsonr(df['clock_ns'], X_pca[:, 0])
print(f'Pearson r(utilization, PC1) = {r_util_pc1:.3f}')
print(f'Pearson r(clock_ns,    PC1) = {r_clock_pc1:.3f}')
print(f'→ PC1 is mostly driven by: {"utilization" if abs(r_util_pc1) > abs(r_clock_pc1) else "clock_ns"}')

---
## Section E2 — Phase 2: clustering + factor-separability tests

CLAUDE.md Phase 2 is very specific about how to answer *"which design knobs
create real distributional separation in feature space?"*. Rather than
eyeballing the PCA scatter, we run four complementary tests and only trust
a factor when several agree:

- **E2.1 k-means k-selection.** Fit k-means for `k in [2, 15]` and pick k
  where at least two of `silhouette`, `Davies-Bouldin`, and the gap
  statistic agree. Store the resulting labels as `df['cluster_id']` —
  these are the labels the Dirichlet-over-clusters partitioner in
  Section F will consume.
- **E2.2 HDBSCAN on UMAP.** A shape-agnostic robustness check on the
  k-means labels. Optional (skipped if the two libraries are missing).
- **E2.3 ARI / NMI / LR-accuracy per factor.** Every metadata factor gets
  scored against the k-means labels (ARI, NMI) *and* against a
  logistic-regression trained on the PCs (5-fold CV accuracy). The
  pre-committed rule from the imports cell says: a factor is a valid
  partition axis iff `ARI >= ARI_MIN` **and** `LR_acc >= LR_ACC_MIN`.
- **E2.4 PERMANOVA.** A pseudo-F test on Euclidean distances in PCA
  space with a permutation null. Complementary evidence that factor
  levels have significantly different centroids.

The exit deliverable of this section is a printed table telling us which
metadata knobs pass the pre-registered rule. Sections F and G refer back
to this table when deciding which partitioners are worth running.


In [ ]:
# -----------------------------------------------------------------
# E2.1 — Pick k for k-means by triangulating THREE quality indices:
#
#     silhouette (higher = better)        - well-separated, tight clusters
#     Davies-Bouldin (LOWER  = better)    - low within/between-scatter ratio
#     gap statistic (higher = better)     - beats a uniform-noise reference
#
# We only *use* a k as the Phase-2 label source when at least two of the
# three indices point to it. That guards against picking a k just because
# one metric flatters it — a common failure mode when the data has no
# real cluster structure.
#
# Silhouette is O(N^2) so we compute it on a fixed random subsample.
# -----------------------------------------------------------------
KM_SUBSAMPLE = min(3000, X_pca.shape[0])
_rng_sub = np.random.default_rng(0)
_sub_idx = _rng_sub.choice(X_pca.shape[0], KM_SUBSAMPLE, replace=False)

def _gap_statistic(X: np.ndarray, k: int,
                   n_refs: int = 3, seed: int = 0) -> float:
    """Tibshirani et al. gap statistic between the data's inertia at k
    and the inertia at k for n_refs uniform reference draws.
    """
    rng = np.random.default_rng(seed)
    km = KMeans(n_clusters=k, n_init=5, random_state=seed).fit(X)
    real_inertia = km.inertia_
    lo, hi = X.min(axis=0), X.max(axis=0)
    ref_inertias = []
    for _ in range(n_refs):
        ref = rng.uniform(lo, hi, size=X.shape)
        km_r = KMeans(n_clusters=k, n_init=3, random_state=seed).fit(ref)
        ref_inertias.append(km_r.inertia_)
    return float(np.log(np.mean(ref_inertias)) - np.log(real_inertia))


sil_scores, db_scores, gap_scores = {}, {}, {}
for k in KMEANS_K_RANGE:
    km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(X_pca)
    labels = km.labels_
    sil_scores[k] = silhouette_score(X_pca[_sub_idx], labels[_sub_idx])
    db_scores[k]  = davies_bouldin_score(X_pca, labels)
    gap_scores[k] = _gap_statistic(X_pca[_sub_idx], k=k, n_refs=3, seed=0)

sil_k = max(sil_scores,   key=sil_scores.get)
db_k  = min(db_scores,    key=db_scores.get)  # DB is minimised
gap_k = max(gap_scores,   key=gap_scores.get)
votes = pd.Series([sil_k, db_k, gap_k]).value_counts()
best_k = int(votes.idxmax()) if votes.iloc[0] >= 2 else sil_k
print(f'Silhouette picks k={sil_k}, Davies-Bouldin picks k={db_k}, '
      f'gap picks k={gap_k}  ->  agreed best_k = {best_k}')

best_km = KMeans(n_clusters=best_k, n_init=25, random_state=0).fit(X_pca)
best_labels = best_km.labels_
df['cluster_id'] = best_labels

fig, axes = plt.subplots(1, 3, figsize=(21, 5))
for ax, (title, scores) in zip(
    axes, [('Silhouette (higher better)', sil_scores),
           ('Davies-Bouldin (lower better)', db_scores),
           ('Gap statistic (higher better)', gap_scores)]):
    ks   = list(scores.keys())
    vals = list(scores.values())
    ax.plot(ks, vals, 'o-')
    ax.axvline(best_k, color='red', linestyle='--',
               linewidth=1, label=f'chosen k={best_k}')
    ax.set_xlabel('k')
    ax.set_title(title)
    ax.legend()
plt.suptitle('Section E2 - k-selection agreement',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# -----------------------------------------------------------------
# E2.2 — HDBSCAN on a UMAP embedding as a robustness cross-check.
#
# k-means assumes k round Gaussian blobs of comparable size. If the true
# structure is stringy or highly imbalanced, k-means silhouette can look
# fine but the labels are meaningless. HDBSCAN + UMAP is agnostic to
# blob shape and produces a "noise" label (-1) for points that don't
# belong to any tight cluster.
#
# If UMAP or HDBSCAN are not installed we skip this cell without failing.
# -----------------------------------------------------------------
if HAS_UMAP and HAS_HDBSCAN:
    reducer  = umap.UMAP(n_neighbors=30, min_dist=0.05,
                         n_components=2, random_state=0)
    X_umap   = reducer.fit_transform(X_pca)
    clusterer = hdbscan.HDBSCAN(min_cluster_size=200, min_samples=25)
    hdb_labels = clusterer.fit_predict(X_umap)

    n_hdb_clusters = int((np.unique(hdb_labels) >= 0).sum())
    n_noise = int((hdb_labels == -1).sum())
    print(f'HDBSCAN-on-UMAP found {n_hdb_clusters} clusters '
          f'({n_noise} / {len(hdb_labels)} points labelled noise).')

    ari_kmeans_hdb = adjusted_rand_score(best_labels, hdb_labels)
    nmi_kmeans_hdb = normalized_mutual_info_score(best_labels, hdb_labels)
    print(f'Cluster-agreement (k-means vs HDBSCAN):  ARI = {ari_kmeans_hdb:.3f}, '
          f'NMI = {nmi_kmeans_hdb:.3f}')

    fig, ax = plt.subplots(figsize=(8, 6))
    cmap_h = mcm.get_cmap('tab20', max(hdb_labels) + 2)
    for lab in np.unique(hdb_labels):
        m = hdb_labels == lab
        color = 'lightgrey' if lab == -1 else cmap_h(lab)
        ax.scatter(X_umap[m, 0], X_umap[m, 1], s=3, alpha=0.4,
                   color=color, label=('noise' if lab == -1 else f'C{lab}'))
    ax.set_title('Section E2 - HDBSCAN clusters on UMAP embedding')
    ax.legend(fontsize=7, markerscale=3, loc='best')
    plt.tight_layout()
    plt.show()
else:
    print('umap-learn / hdbscan not installed -> skipping E2.2 robustness check.')


In [ ]:
# -----------------------------------------------------------------
# E2.3 — Score every metadata factor with ARI, NMI, and a
# logistic-regression-from-PCs accuracy.
#
# For each factor F we compare against the best-k k-means labels:
#   - ARI(clusters, F) : how much of F's structure the unsupervised
#                        partition already recovers.
#   - NMI(clusters, F) : same idea, information-theoretic.
#   - LR-acc(F | PCs)  : 5-fold cross-validated accuracy of a
#                        multinomial logistic regression trained on the
#                        PCA scores to predict F.
#
# CLAUDE.md's pre-committed rule:
#     "A metadata factor is a valid partition axis iff
#      ARI >= ARI_MIN  AND  LR_acc >= LR_ACC_MIN."
# We literally apply that rule in the last column of the printed table.
# -----------------------------------------------------------------
FACTORS = ['design_name', 'macro_count', 'macro_placement',
           'power_mesh', 'filler_insertion',
           'clock_bin', 'util_bin']

# Bin continuous knobs into 3 quantile levels so ARI/LR see a categorical target.
df['clock_bin'] = pd.qcut(df['clock_ns'],   q=3, duplicates='drop').astype(str)
df['util_bin']  = pd.qcut(df['utilization'], q=3, duplicates='drop').astype(str)

factor_rows = []
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
for f in FACTORS:
    y = df[f].astype(str).values
    if len(np.unique(y)) < 2:
        continue
    ari = adjusted_rand_score(y, best_labels)
    nmi = normalized_mutual_info_score(y, best_labels)
    try:
        lr_acc = cross_val_score(
            LogisticRegression(max_iter=500, multi_class='auto', n_jobs=None),
            X_pca, y, cv=skf, scoring='accuracy', n_jobs=None,
        ).mean()
    except Exception:
        lr_acc = np.nan
    verdict = ('PASS' if (ari >= ARI_MIN and lr_acc >= LR_ACC_MIN)
               else 'weak')
    factor_rows.append({'factor': f, 'ARI': ari, 'NMI': nmi,
                        'LR_acc': lr_acc, 'verdict': verdict})

factor_df = (pd.DataFrame(factor_rows)
               .sort_values('LR_acc', ascending=False)
               .reset_index(drop=True))
print(f'Factor-separability table (pre-committed thresholds '
      f'ARI >= {ARI_MIN}, LR_acc >= {LR_ACC_MIN}):')
print(factor_df.round(3).to_string(index=False))

# ---- companion visual: bar chart of LR_acc with threshold line ----
fig, ax = plt.subplots(figsize=(11, 4))
colors = ['#4caf50' if v == 'PASS' else '#ff9800' for v in factor_df['verdict']]
ax.bar(factor_df['factor'], factor_df['LR_acc'],
       color=colors, alpha=0.85, edgecolor='white')
ax.axhline(LR_ACC_MIN, color='red', linestyle='--',
           linewidth=1, label=f'LR_ACC_MIN = {LR_ACC_MIN}')
ax.set_ylim(0, 1)
ax.set_ylabel('LR accuracy (5-fold CV)')
ax.set_title('Section E2 — factor separability from PCA scores')
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# -----------------------------------------------------------------
# E2.4 — PERMANOVA-style significance of each metadata factor.
#
# PERMANOVA asks: is the mean position of PCA points different across
# levels of the factor by more than we would see under a random shuffle?
#
# We implement the pseudo-F on squared Euclidean distances in PCA space
# with a permutation null. It is complementary to ARI/LR: ARI/LR say
# "the factor can be reconstructed from the descriptors"; PERMANOVA
# says "the centroids for each factor level are significantly apart."
# -----------------------------------------------------------------
def permanova_pseudo_f(X: np.ndarray,
                       labels: np.ndarray,
                       n_perm: int = 199,
                       seed: int = 0) -> tuple:
    """Return (pseudo_F, p_value) for a one-way PERMANOVA.

    Uses squared Euclidean distances -> equivalent to ANOVA on the sum of
    squared centroid distances, which is O(N * k) instead of O(N^2).
    """
    rng = np.random.default_rng(seed)
    labels = np.asarray(labels)
    N = X.shape[0]
    grand_centroid = X.mean(axis=0)
    ss_total = float(((X - grand_centroid) ** 2).sum())

    def _ss_between(lab):
        ss = 0.0
        for lv in np.unique(lab):
            m = lab == lv
            if m.sum() == 0:
                continue
            centroid = X[m].mean(axis=0)
            ss += m.sum() * float(((centroid - grand_centroid) ** 2).sum())
        return ss

    ss_between = _ss_between(labels)
    ss_within  = ss_total - ss_between
    a = len(np.unique(labels))
    pseudo_F = (ss_between / (a - 1)) / (ss_within / (N - a)) if a > 1 else np.nan

    # Permutation null
    ge = 0
    for _ in range(n_perm):
        perm = rng.permutation(labels)
        ss_b_p = _ss_between(perm)
        ss_w_p = ss_total - ss_b_p
        F_p = (ss_b_p / (a - 1)) / (ss_w_p / (N - a)) if a > 1 else 0.0
        if F_p >= pseudo_F:
            ge += 1
    p_val = (ge + 1) / (n_perm + 1)
    return pseudo_F, p_val


PERM_FACTORS = ['design_name', 'macro_count', 'macro_placement',
                'power_mesh', 'filler_insertion',
                'clock_bin_perm', 'util_bin_perm']

# Bin the continuous knobs so PERMANOVA sees categorical levels.
df['util_bin_perm']  = pd.qcut(df['utilization'], q=3, duplicates='drop').astype(str)
df['clock_bin_perm'] = pd.cut(df['clock_ns'], bins=[0, 3, 10, 25]).astype(str)

perm_rows = []
for f in PERM_FACTORS:
    F, p = permanova_pseudo_f(X_pca, df[f].values, n_perm=99, seed=0)
    perm_rows.append({'factor': f, 'pseudoF': F, 'p_value': p,
                      'sig': '***' if p < 0.001 else '*' if p < 0.05 else 'n.s.'})
perm_df = pd.DataFrame(perm_rows).sort_values('pseudoF', ascending=False)
print('PERMANOVA (pseudo-F on squared Euclidean distances in PCA space):')
print(perm_df.round(3).to_string(index=False))


---
## Section F — Partition-Strategy Validation

Applies the seven partitioners listed in CLAUDE.md Phase 3 and scores each
with **four** complementary metrics so the picture is not one-dimensional.

### Metrics
- **KS statistic** — max gap between the CDFs of a per-channel scalar
  distribution across two parties. 0 = identical.
- **Wasserstein distance (EMD)** — average "transport cost" between the
  same two distributions. Sensitive to shifts KS misses.
- **JS divergence** — symmetric information-theoretic distance between the
  DRC severity-tier histograms of two parties. This is the label-side
  metric CLAUDE.md asks for.
- **MMD (RBF, permutation-tested)** — a multivariate test on the full
  descriptor block. Flags feature-level shift that per-channel KS would
  aggregate away.

### Strategies covered
1. **IID** — stratified round-robin baseline.
2. **Real per-pixel Noise** — replaces v1's ad-hoc scalar simulation.
   For each noise level we sample a small subset of files per party,
   inject `N(0, σ)` into each feature map, and *recompute the
   descriptor block on the noised arrays*. This measures the real
   effect of the noise on the statistics we actually feed downstream.
3. **Synthetic grid** — quantile grid over the axes E2 blessed as
   PASS.
4. **Design-held-out** — one design per party (or grouped when
   `n_designs > N_PARTIES`); yields extreme design-identity skew.
5. **PPA persona** — low-power (low freq + low util), high-perf (high
   freq), area-driven (high util) — the "realistic EDA-team" partition
   framing.
6. **Flow-recipe** — power-mesh x filler x macro-placement — captures
   pure downstream-flow diversity, expected to be near-IID.
7. **Dirichlet(β) over discovered cluster labels** for
   β ∈ {0.1, 0.5, 5.0} — β -> 0 makes labels heavily skewed, β large
   recovers IID. Uses the `cluster_id` column produced in E2.


In [ ]:
N_PARTIES = 6

# --------------------------------------------------------------------
# Which per-channel scalar do we test with KS / Wasserstein? Historically
# it was "the channel mean", but we can now pick the two channels whose
# means have the largest cross-sample variance so KS has something to
# distinguish. Same as the v1 behaviour, just made explicit.
# --------------------------------------------------------------------
channel_var_vals = {ch: df[f'{ch}_mean'].var() for ch in CHANNEL_NAMES}
top2_channels    = sorted(channel_var_vals, key=channel_var_vals.get,
                          reverse=True)[:2]
print(f'Highest-variance channels (validation proxies): {top2_channels}')


# ====================================================================
# Univariate: KS and Wasserstein on each channel_mean, averaged over
# every unordered pair of parties.
# ====================================================================
def pairwise_ks_wasserstein(partitions, channels):
    ks_all, wass_all = [], []
    per_ch_ks = {ch: [] for ch in channels}
    for i in range(len(partitions)):
        for j in range(i + 1, len(partitions)):
            for ch in channels:
                col = f'{ch}_mean'
                a = partitions[i][col].dropna().values
                b = partitions[j][col].dropna().values
                if len(a) > 1 and len(b) > 1:
                    ks = ks_2samp(a, b).statistic
                    w  = wasserstein_distance(a, b)
                    ks_all.append(ks)
                    wass_all.append(w)
                    per_ch_ks[ch].append(ks)
    return (float(np.mean(ks_all))   if ks_all   else 0.0,
            float(np.mean(wass_all)) if wass_all else 0.0,
            per_ch_ks)


# ====================================================================
# Label-side JS divergence: how far apart are the DRC-tier histograms
# of the parties, on average across pairs?
# ====================================================================
def pairwise_js_labels(partitions,
                       label_col: str = 'tier',
                       n_tiers: int = 4) -> float:
    """Mean pairwise Jensen-Shannon divergence between per-party
    label-tier histograms. Returns 0 if labels are missing.
    """
    if any(label_col not in p.columns for p in partitions):
        return float('nan')
    hists = []
    for p in partitions:
        vals = p[label_col].dropna().astype(int).values
        h, _ = np.histogram(vals, bins=np.arange(n_tiers + 1))
        h = h.astype(np.float64)
        if h.sum() == 0:
            h[:] = 1.0
        hists.append(h / h.sum())
    js_vals = []
    for i in range(len(hists)):
        for j in range(i + 1, len(hists)):
            # scipy returns the JS *distance* (sqrt of divergence), so
            # square it back to the divergence in [0, ln2].
            js_vals.append(jensenshannon(hists[i], hists[j]) ** 2)
    return float(np.mean(js_vals)) if js_vals else 0.0


# ====================================================================
# Multivariate MMD^2 with an RBF kernel, plus a permutation p-value.
# Sub-samples each party to at most MMD_SUBSAMPLE for tractability.
# ====================================================================
MMD_SUBSAMPLE = 400
MMD_PERM      = 200

def _rbf_gram(X: np.ndarray, Y: np.ndarray, sigma: float) -> np.ndarray:
    D2 = ((X[:, None, :] - Y[None, :, :]) ** 2).sum(axis=-1)
    return np.exp(-D2 / (2.0 * sigma ** 2))

def _mmd2(X: np.ndarray, Y: np.ndarray, sigma: float) -> float:
    Kxx = _rbf_gram(X, X, sigma)
    Kyy = _rbf_gram(Y, Y, sigma)
    Kxy = _rbf_gram(X, Y, sigma)
    return float(Kxx.mean() + Kyy.mean() - 2.0 * Kxy.mean())

def pairwise_mmd(partitions, feat_cols=None,
                 n_perm: int = MMD_PERM, seed: int = 0) -> tuple:
    """Mean MMD^2 and mean permutation p-value across all party pairs.

    Uses the median heuristic to pick the RBF bandwidth on each pair.
    """
    if feat_cols is None:
        feat_cols = mean_cols
    rng = np.random.default_rng(seed)
    mmd_vals, p_vals = [], []
    for i in range(len(partitions)):
        for j in range(i + 1, len(partitions)):
            A = partitions[i][feat_cols].dropna().values
            B = partitions[j][feat_cols].dropna().values
            if len(A) < 20 or len(B) < 20:
                continue
            idx_A = rng.choice(len(A), min(MMD_SUBSAMPLE, len(A)), replace=False)
            idx_B = rng.choice(len(B), min(MMD_SUBSAMPLE, len(B)), replace=False)
            A, B = A[idx_A], B[idx_B]
            # Median-heuristic bandwidth.
            Z = np.vstack([A, B])
            D2 = ((Z[:, None, :] - Z[None, :, :]) ** 2).sum(axis=-1)
            sigma = np.sqrt(np.median(D2[D2 > 0]) / 2.0)
            observed = _mmd2(A, B, sigma)

            # Permutation null.
            merged = np.vstack([A, B])
            n = len(A)
            ge = 0
            for _ in range(n_perm):
                rng.shuffle(merged)
                Ap, Bp = merged[:n], merged[n:]
                if _mmd2(Ap, Bp, sigma) >= observed:
                    ge += 1
            mmd_vals.append(observed)
            p_vals.append((ge + 1) / (n_perm + 1))
    return (float(np.mean(mmd_vals)) if mmd_vals else 0.0,
            float(np.mean(p_vals))   if p_vals   else 1.0)


# --------------------------------------------------------------------
# Master scoring function used by every partitioner block below.
# Returns a flat dict so results tabulate cleanly at the end.
# --------------------------------------------------------------------
def score_partition(name: str, parts) -> dict:
    ks, wass, _ = pairwise_ks_wasserstein(parts, CHANNEL_NAMES)
    # JS on labels requires each partition to carry the `tier` column.
    if 'df_label' in globals() and 'tier' in df_label.columns:
        parts_with_label = [p.merge(df_label[['filename', 'tier']],
                                    on='filename', how='left')
                            for p in parts]
        js = pairwise_js_labels(parts_with_label)
    else:
        js = float('nan')
    mmd, mmd_p = pairwise_mmd(parts, feat_cols=mean_cols, n_perm=100, seed=0)
    return {
        'strategy':      name,
        'sizes':         [len(p) for p in parts],
        'mean_ks':       ks,
        'mean_wass':     wass,
        'mean_js_label': js,
        'mean_mmd':      mmd,
        'mmd_p':         mmd_p,
        'mmd_signif':    mmd_p < MMD_P,
    }


print('Helper functions defined.')
print(f'  MMD subsample per party: {MMD_SUBSAMPLE}  (permutations: {MMD_PERM})')


In [ ]:
# ---- F1: IID partitioning ----
iid_parts = IIDPartitioner(n_partitions=N_PARTIES, seed=42).partition(df)
iid_sizes = [len(p) for p in iid_parts]
iid_ks, iid_wass, iid_per_ch = pairwise_ks_wasserstein(iid_parts, CHANNEL_NAMES)

print(f'IID partition sizes: {iid_sizes}')
print(f'IID mean pairwise KS:          {iid_ks:.4f}')
print(f'IID mean pairwise Wasserstein: {iid_wass:.6f}')

# KS distribution across channels
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Section F1 — IID Partitioning Baseline', fontsize=13, fontweight='bold')

ax = axes[0]
ks_per_ch_mean = [np.mean(iid_per_ch[ch]) if iid_per_ch[ch] else 0 for ch in CHANNEL_NAMES]
bars = ax.bar(range(N_CHANNELS), ks_per_ch_mean,
              color=[palette(i) for i in range(N_CHANNELS)], alpha=0.85, edgecolor='white')
ax.set_xticks(range(N_CHANNELS))
ax.set_xticklabels(CHANNEL_NAMES, rotation=45, ha='right', fontsize=8)
ax.set_title('Mean pairwise KS per channel (IID)\nShould be near 0')
ax.set_ylabel('Mean KS statistic')
ax.axhline(0.05, color='red', linestyle='--', linewidth=1, label='KS=0.05')
ax.legend(fontsize=9)

# Utilization distribution per party
ax = axes[1]
for i, p in enumerate(iid_parts):
    ax.hist(p['utilization'].values, bins=15, alpha=0.5,
            color=mcm.get_cmap('tab10')(i / 10), label=f'Party {i}',
            density=True, histtype='stepfilled', edgecolor='white')
ax.set_title('utilization distribution per party (IID)')
ax.set_xlabel('utilization')
ax.set_ylabel('Density')
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# =====================================================================
# F2 — REAL per-pixel noise-based partitioning
#
# CLAUDE.md's rewrite of v1's noise cell: instead of adding Gaussian
# noise directly to the *scalar* per-sample channel means (which the
# review flagged as circular), we actually reload a small subsample of
# feature maps per party, inject N(0, sigma) on the per-pixel arrays,
# and recompute the descriptor block on the noised maps. Only then do
# we score the partition with KS / Wasserstein.
#
# The subsample size trades runtime for statistical resolution -- 200
# files per party per noise level is plenty for KS to reach ~0.03 std
# under IID, comfortably below the NOISE_KS_FLOOR threshold.
# =====================================================================
NOISE_STDS      = [0.0, 0.02, 0.05, 0.10, 0.20]
NOISE_SUBSAMPLE = 200           # files per party actually loaded
noise_rng       = np.random.default_rng(42)


def _noise_descriptors(files_subset, sigma: float) -> pd.DataFrame:
    """Load a small list of feature files, inject N(0, sigma) per pixel,
    recompute the descriptor block on the noised arrays, and return the
    corresponding DataFrame (same schema as `df[FEAT_COLS]`).
    """
    rows = []
    for fname in files_subset:
        fpath = os.path.join(FEATURE_DIR, fname)
        try:
            arr = np.load(fpath).astype(np.float32)
            if sigma > 0:
                arr = arr + noise_rng.normal(0.0, sigma, arr.shape).astype(np.float32)
                arr = np.clip(arr, 0.0, None)     # feature maps are non-negative
            rows.append(extract_channel_stats(arr))
        except Exception:
            continue
    if not rows:
        return pd.DataFrame(columns=FEAT_COLS)
    return pd.DataFrame(rows, columns=FEAT_COLS)


noise_results = []
for sigma in NOISE_STDS:
    parts = NoiseFeaturePartitioner(
        n_partitions=N_PARTIES, noise_std_min=0.0,
        noise_std_max=sigma, seed=42,
    ).partition(df)

    noised_parts = []
    for p in parts:
        # Random subsample of this party's filenames -> reload -> add noise.
        take = min(NOISE_SUBSAMPLE, len(p))
        sub_names = noise_rng.choice(p['filename'].values, take, replace=False)
        noised_stats = _noise_descriptors(sub_names, sigma)
        merged = pd.DataFrame({'filename': sub_names})
        for col in FEAT_COLS:
            merged[col] = noised_stats[col].values if col in noised_stats else np.nan
        # Round-trip metadata so downstream code sees the same columns.
        noised_parts.append(
            merged.merge(df_meta, on='filename', how='left').dropna(subset=mean_cols)
        )

    ks, wass, _ = pairwise_ks_wasserstein(noised_parts, CHANNEL_NAMES)
    noise_results.append({
        'sigma':     sigma,
        'mean_ks':   ks,
        'mean_wass': wass,
    })

noise_df = pd.DataFrame(noise_results)
print('Real per-pixel noise sweep (descriptors recomputed on noised maps):')
print(noise_df.round(5).to_string(index=False))

# Recommended sigma: first level where KS clears BOTH the absolute floor
# and the "2x IID baseline" bar (iid_ks comes from cell F1).
target       = max(NOISE_KS_FLOOR, 2.0 * iid_ks)
above_target = noise_df[noise_df['mean_ks'] >= target]
rec_noise    = float(above_target.iloc[0]['sigma']) if not above_target.empty else None

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Section F2 - Real per-pixel Noise partitioning',
             fontsize=13, fontweight='bold')

ax = axes[0]
ax.plot(noise_df['sigma'], noise_df['mean_ks'], 'bo-', linewidth=2, markersize=7)
ax.axhline(iid_ks, color='green', linestyle='--', linewidth=1.5,
           label=f'IID baseline KS = {iid_ks:.3f}')
ax.axhline(target, color='red', linestyle=':', linewidth=1,
           label=f'meaningful threshold = {target:.3f}')
ax.set_title('KS(channel means) vs noise sigma')
ax.set_xlabel('sigma')
ax.set_ylabel('Mean pairwise KS')
ax.legend()

ax = axes[1]
ax.plot(noise_df['sigma'], noise_df['mean_wass'], 'rs-', linewidth=2, markersize=7)
ax.axhline(iid_wass, color='green', linestyle='--', linewidth=1.5,
           label=f'IID baseline W = {iid_wass:.5f}')
ax.set_title('Wasserstein vs noise sigma')
ax.set_xlabel('sigma')
ax.set_ylabel('Mean pairwise Wasserstein')
ax.legend()

plt.tight_layout()
plt.show()

if rec_noise is not None:
    print(f'\nRecommended noise sigma for meaningful feature Non-IID: {rec_noise}')
    print(f'  (first level where KS >= {target:.3f} = max(NOISE_KS_FLOOR, 2x IID))')
else:
    print(f'\nKS never reached {target:.3f} in the tested sigma range.')


In [ ]:
# ---- F3: Synthetic feature partitioning (utilization × clock_ns grid) ----
synth_parts = SyntheticFeaturePartitioner(
    n_partitions=N_PARTIES,
    feature_col_x='utilization',
    feature_col_y='clock_ns',
    seed=42,
).partition(df)

synth_sizes = [len(p) for p in synth_parts]
synth_ks, synth_wass, synth_per_ch = pairwise_ks_wasserstein(synth_parts, CHANNEL_NAMES)

print(f'Synthetic partition sizes: {synth_sizes}')
print(f'Synthetic mean pairwise KS:          {synth_ks:.4f}')
print(f'Synthetic mean pairwise Wasserstein: {synth_wass:.6f}')
print(f'\nKS ratio vs IID baseline: {synth_ks / (iid_ks + 1e-9):.1f}×')

fig, axes = plt.subplots(1, 3, figsize=(21, 6))
fig.suptitle('Section F3 — Synthetic Feature Partitioning (utilization × clock_ns grid)',
             fontsize=13, fontweight='bold')

# Feature-space scatter with party assignment
ax = axes[0]
cmap10 = mcm.get_cmap('tab10')
for i, p in enumerate(synth_parts):
    ax.scatter(p['utilization'], p['clock_ns'],
               s=5, alpha=0.4, color=cmap10(i/10), label=f'Party {i}')
ax.set_xlabel('utilization')
ax.set_ylabel('clock_ns')
ax.set_title('Party assignment in (utilization, clock_ns) space')
ax.legend(fontsize=8, markerscale=4)

# Per-channel KS comparison: IID vs Synthetic
ax = axes[1]
x = np.arange(N_CHANNELS)
w = 0.35
iid_ks_vals   = [np.mean(iid_per_ch[ch])   if iid_per_ch[ch]   else 0 for ch in CHANNEL_NAMES]
synth_ks_vals = [np.mean(synth_per_ch[ch]) if synth_per_ch[ch] else 0 for ch in CHANNEL_NAMES]
ax.bar(x - w/2, iid_ks_vals,   width=w, label='IID',       color='steelblue', alpha=0.8, edgecolor='white')
ax.bar(x + w/2, synth_ks_vals, width=w, label='Synthetic', color='darkorange',alpha=0.8, edgecolor='white')
ax.set_xticks(x)
ax.set_xticklabels(CHANNEL_NAMES, rotation=45, ha='right', fontsize=8)
ax.set_title('Mean pairwise KS: IID vs Synthetic')
ax.set_ylabel('Mean KS statistic')
ax.legend(fontsize=9)

# Box plots of top-variance channel per party
ax = axes[2]
key_ch = top2_channels[0]
key_col = f'{key_ch}_mean'
data_synth = [p[key_col].values for p in synth_parts]
bp = ax.boxplot(data_synth, patch_artist=True, notch=False,
                medianprops={'color':'black','linewidth':1.4},
                flierprops={'marker':'.','markersize':2,'alpha':0.3})
for patch, i in zip(bp['boxes'], range(N_PARTIES)):
    patch.set_facecolor(cmap10(i/10))
    patch.set_alpha(0.75)
ax.set_xticks(range(1, N_PARTIES+1))
ax.set_xticklabels([f'P{i}' for i in range(N_PARTIES)])
ax.set_title(f'{key_ch} distribution per party (Synthetic)')
ax.set_ylabel(f'{key_ch}_mean')

plt.tight_layout()
plt.show()

In [ ]:
# =====================================================================
# F5 — Design-held-out partition (leave-one-design-per-party).
#
# CLAUDE.md's strategy (b): each party sees exactly one design (or a
# balanced group of designs when n_designs > N_PARTIES). Feature and
# label distributions should shift more drastically than under any
# knob-based split, which is exactly what makes it the "hard extreme"
# for FL comparisons.
# =====================================================================
groups = np.array_split(designs_uniq, N_PARTIES)
design_groups = [list(g) for g in groups]
print('Design-held-out groups:')
for i, g in enumerate(design_groups):
    print(f'  Party {i}: {g}')

design_parts = [
    df[df['design_name'].isin(g)].reset_index(drop=True)
    for g in design_groups
]
print(f'Design-held-out partition sizes: {[len(p) for p in design_parts]}')

# NEW: hold the results in a growing list of dicts we'll consolidate at F9.
full_scores = [score_partition('Design-held-out', design_parts)]


In [ ]:
# =====================================================================
# F6 — PPA persona partition.
#
# CLAUDE.md's "realistic EDA-team" framing: three archetype personas
# grounded in the dataset's own knobs.
#   - low-power:       long clock  (>= 10 ns)  AND  low  utilization (<= 0.75)
#   - high-performance: short clock (<= 3 ns)
#   - area-driven:     high utilization (>= 0.85)
# Samples that fit none of these buckets fall into a "generic" bucket
# so no sample is dropped. We then round-robin each bucket into parties
# to reach N_PARTIES; typically two low-power + two high-perf + two
# area-driven parties.
# =====================================================================
def _persona(row):
    if row['clock_ns'] >= 10 and row['utilization'] <= 0.75:
        return 'low_power'
    if row['clock_ns'] <= 3:
        return 'high_perf'
    if row['utilization'] >= 0.85:
        return 'area_driven'
    return 'generic'

df['persona'] = df.apply(_persona, axis=1)
persona_counts = df['persona'].value_counts()
print('Persona distribution:')
print(persona_counts.to_string())

persona_parts = []
personas_present = [p for p in ['low_power', 'high_perf',
                                'area_driven', 'generic']
                    if (df['persona'] == p).any()]
per_persona = max(1, N_PARTIES // len(personas_present))

pid = 0
for pname in personas_present:
    sub = df[df['persona'] == pname].sample(frac=1.0, random_state=0)
    chunks = np.array_split(sub, per_persona)
    for ch in chunks:
        if pid < N_PARTIES:
            persona_parts.append(ch.reset_index(drop=True))
            pid += 1
# If personas x per_persona < N_PARTIES, absorb any remainder into
# the last party to keep the party count exactly N_PARTIES.
while len(persona_parts) < N_PARTIES:
    persona_parts.append(pd.DataFrame(columns=df.columns))

print(f'Persona partition sizes: {[len(p) for p in persona_parts]}')
full_scores.append(score_partition('PPA persona', persona_parts))


In [ ]:
# =====================================================================
# F7 — Flow-recipe partition.
#
# Combines (power_mesh, filler_insertion, macro_placement). CLAUDE.md
# predicts these three knobs move together with routing flow choices
# rather than with the physical difficulty of the design -> expected
# to be near-IID in feature space and mildly non-IID on labels.
# Included precisely so we can show it *is* near-IID (a negative control).
# =====================================================================
recipe_key = (df['power_mesh'].astype(str)
              + '_' + df['filler_insertion'].astype(str)
              + '_' + df['macro_placement'].astype(str))
recipes    = recipe_key.value_counts().index.tolist()

# Balance recipes across parties: sort recipes by size, assign in a
# largest-first snake pattern so each party gets a mix of large & small.
sorted_recipes = sorted(recipes, key=lambda r: -recipe_key.value_counts()[r])
recipe_to_party = {}
for i, r in enumerate(sorted_recipes):
    lap = i // N_PARTIES
    pos = i % N_PARTIES
    recipe_to_party[r] = pos if lap % 2 == 0 else (N_PARTIES - 1 - pos)

flow_parts = [
    df[recipe_key.map(recipe_to_party) == i].reset_index(drop=True)
    for i in range(N_PARTIES)
]
print(f'Flow-recipe partition sizes: {[len(p) for p in flow_parts]}')
full_scores.append(score_partition('Flow-recipe', flow_parts))


In [ ]:
# =====================================================================
# F8 — Dirichlet(beta) over the k-means cluster_id from Section E2.
#
# For beta in {0.1, 0.5, 5.0} we draw a Dirichlet mixture per cluster
# and assign samples proportionally. Small beta -> heavy label
# imbalance across parties; large beta -> near-IID. This is the
# CLAUDE.md-mandated "Dirichlet over discovered cluster labels" scheme.
# =====================================================================
BETAS = [0.1, 0.5, 5.0]
dirichlet_parts_by_beta = {}
for beta in BETAS:
    parts_dir = DirichletLabelPartitioner(
        n_partitions=N_PARTIES, alpha=beta,
        label_col='cluster_id', seed=42,
    ).partition(df)
    dirichlet_parts_by_beta[beta] = parts_dir
    s = score_partition(f'Dirichlet(cluster, beta={beta})', parts_dir)
    print(f'  beta={beta}  sizes={s["sizes"]}  '
          f'KS={s["mean_ks"]:.4f}  JS={s["mean_js_label"]:.4f}  '
          f'MMD_p={s["mmd_p"]:.3f}')
    full_scores.append(s)


In [ ]:
# =====================================================================
# F9 — Consolidated multi-metric scoring table.
#
# The individual F5..F8 blocks above only recorded strategies to
# `full_scores`. Here we add the earlier strategies (IID, real Noise,
# Synthetic grid), print the four-metric table, and produce the
# "KS vs JS on labels" scatter that lets us pick a strategy on more
# than one dimension.
# =====================================================================
full_scores.insert(0, score_partition('IID (baseline)', iid_parts))
full_scores.insert(1, score_partition('Synthetic util x clock', synth_parts))
if rec_noise is not None:
    # Score the recommended noise level once more using the same
    # multi-metric helper for apples-to-apples comparison.
    parts_noise = NoiseFeaturePartitioner(
        n_partitions=N_PARTIES, noise_std_min=0.0,
        noise_std_max=rec_noise, seed=42,
    ).partition(df)
    full_scores.insert(2, score_partition(f'Noise sigma={rec_noise}', parts_noise))

scores_df = pd.DataFrame(full_scores)
print('Multi-metric partition-strategy scoreboard:')
print(scores_df[['strategy', 'mean_ks', 'mean_wass',
                 'mean_js_label', 'mean_mmd', 'mmd_p', 'mmd_signif']]
      .round(4).to_string(index=False))

fig, ax = plt.subplots(figsize=(11, 6))
for _, r in scores_df.iterrows():
    color = 'green'  if r['mmd_signif'] else 'grey'
    ax.scatter(r['mean_ks'], r['mean_js_label'],
               s=80, color=color, edgecolor='black', alpha=0.85)
    ax.annotate(r['strategy'], (r['mean_ks'], r['mean_js_label']),
                textcoords='offset points', xytext=(6, 4), fontsize=8)
ax.set_xlabel('Mean pairwise KS  (feature-side Non-IID)')
ax.set_ylabel('Mean pairwise JS  (DRC-tier Non-IID)')
ax.set_title('Section F - strategies in the (feature skew, label skew) plane\n'
             'Green = MMD significant at p < MMD_P')
plt.tight_layout()
plt.show()


---
## Section G — Summary and pre-committed-threshold decisions

Every recommendation below is derived by comparing measurements produced
earlier in the notebook against the constants declared in the imports
cell (`ARI_MIN`, `LR_ACC_MIN`, `MMD_P`, `NOISE_KS_FLOOR`). Those constants
were fixed BEFORE seeing any of the numbers, so no threshold has been
retrofitted to make a preferred partitioner look good.


In [ ]:
print('=' * 74)
print('CircuitNet-N28  --  DATA-DRIVEN PARTITIONING SUMMARY')
print('=' * 74)

# ---------- Block 1: pre-committed thresholds -----------------------
print(f'\n0. PRE-COMMITTED THRESHOLDS')
print(f'   ARI_MIN            = {ARI_MIN}   (cluster/factor overlap)')
print(f'   LR_ACC_MIN         = {LR_ACC_MIN}   (LR-from-PCs accuracy)')
print(f'   MMD_P              = {MMD_P}   (MMD permutation p-value)')
print(f'   NOISE_KS_FLOOR     = {NOISE_KS_FLOOR}  (min real-noise KS)')
print(f'   DOMAIN_AUC_MAX     = {DOMAIN_AUC_MAX}  (N14/N45 cross-node)')

# ---------- Block 2: design-space balance ---------------------------
print('\n1. DESIGN PARAMETER BALANCE (min/max sample-count ratio)')
for col in CAT_COLS:
    vc = df_meta[col].value_counts()
    bal = vc.min() / vc.max()
    verdict = 'OK' if bal > 0.5 else 'IMBALANCED'
    print(f'   {col:<22s}: balance={bal:.3f}  [{verdict}]')

# ---------- Block 3: PCA structure ----------------------------------
print('\n2. PCA STRUCTURE (whitened)')
print(f'   Descriptor dimensionality        : {len(FEAT_COLS)}')
print(f'   PCs retained (>= 90% variance)   : {n_for_90}')

# ---------- Block 4: Phase-2 factor-separability verdicts ----------
print('\n3. VALID PARTITION AXES (Phase-2 pre-committed rule)')
if 'factor_df' in globals():
    pass_axes = factor_df[factor_df['verdict'] == 'PASS']
    if pass_axes.empty:
        print('   No metadata factor cleared BOTH ARI and LR thresholds.')
    else:
        for _, r in pass_axes.iterrows():
            print(f'   PASS  {r["factor"]:<18s}  ARI={r["ARI"]:.2f}  '
                  f'NMI={r["NMI"]:.2f}  LR_acc={r["LR_acc"]:.2f}')
    weak_axes = factor_df[factor_df['verdict'] == 'weak']
    for _, r in weak_axes.iterrows():
        reasons = []
        if r['ARI']    < ARI_MIN:   reasons.append(f'ARI<{ARI_MIN}')
        if r['LR_acc'] < LR_ACC_MIN:reasons.append(f'LR_acc<{LR_ACC_MIN}')
        print(f'   weak  {r["factor"]:<18s}  ({", ".join(reasons)})')
else:
    print('   factor_df not defined -- E2.3 was not executed.')

# ---------- Block 5: PERMANOVA cross-check --------------------------
print('\n4. PERMANOVA SIGNIFICANCE (pseudo-F, p<0.05)')
if 'perm_df' in globals():
    sig = perm_df[perm_df['p_value'] < 0.05]
    for _, r in sig.iterrows():
        print(f'   {r["factor"]:<20s}  pseudo_F={r["pseudoF"]:.2f}  '
              f'p={r["p_value"]:.3f}  {r["sig"]}')
else:
    print('   perm_df not defined -- E2.4 was not executed.')

# ---------- Block 6: partitioning scoreboard ------------------------
print('\n5. PARTITION-STRATEGY SCOREBOARD (F9 consolidated)')
if 'scores_df' in globals():
    cols = ['strategy', 'mean_ks', 'mean_wass',
            'mean_js_label', 'mean_mmd', 'mmd_p', 'mmd_signif']
    print(scores_df[cols].round(4).to_string(index=False))

    # Recommended strategy: highest KS among those with mmd_p < MMD_P
    # AND non-trivial JS on labels (>= 0.02). Falls back to "highest KS"
    # if no strategy is significant, and prints the fallback reason.
    candidates = scores_df[(scores_df['mmd_signif'])
                           & (scores_df['mean_js_label'].fillna(0) >= 0.02)]
    if candidates.empty:
        chosen = scores_df.iloc[scores_df['mean_ks'].astype(float).idxmax()]
        why = ('no strategy passed MMD_P and JS>=0.02 filter -> falling '
               'back to max KS')
    else:
        chosen = candidates.iloc[candidates['mean_ks'].astype(float).argmax()]
        why    = f'MMD significant (p < {MMD_P}) and JS on labels >= 0.02'
    print(f'\n   RECOMMENDED strategy : {chosen["strategy"]}')
    print(f'   Reason               : {why}')
    print(f'   Mean KS              : {chosen["mean_ks"]:.4f}')
    print(f'   Mean JS (labels)     : {chosen["mean_js_label"]:.4f}')
    print(f'   Mean MMD (p-value)   : {chosen["mean_mmd"]:.4f}  '
          f'({chosen["mmd_p"]:.3f})')

if rec_noise is not None:
    print(f'\n   Recommended real-noise sigma for a mild feature-only skew: '
          f'{rec_noise}')

print('=' * 74)


In [ ]:
# =====================================================================
# Final visual: every strategy in the (quantity skew, feature skew)
# plane, coloured by whether its MMD-permutation p-value clears MMD_P.
#
# Reading the plot:
#   - Upper-right, GREEN : strong feature Non-IID AND multivariate MMD
#                          is significant -> a genuinely non-IID split.
#   - Upper-right, GREY  : KS looks non-IID but MMD says the shift is
#                          within the noise of a random partition.
#   - Bottom-left        : essentially IID.
# =====================================================================
if 'scores_df' in globals():
    fig, ax = plt.subplots(figsize=(11, 7))
    for _, r in scores_df.iterrows():
        color  = 'green' if r['mmd_signif'] else 'grey'
        marker = 'o'     if 'IID' in r['strategy'] else 's'
        ax.scatter(np.std(r['sizes']), r['mean_ks'],
                   s=110, color=color, marker=marker,
                   edgecolor='black', alpha=0.9)
        ax.annotate(r['strategy'], (np.std(r['sizes']), r['mean_ks']),
                    textcoords='offset points', xytext=(5, 4), fontsize=8)
    ax.set_xlabel('Partition size std  (->  more quantity skew)')
    ax.set_ylabel('Mean pairwise KS  (->  more feature Non-IID)')
    ax.set_title('All strategies in the (quantity skew, feature skew) plane\n'
                 f'Green = MMD-permutation p < MMD_P ({MMD_P})')
    ax.axhline(iid_ks * 2, color='grey', linestyle='--',
               linewidth=0.8, label='2x IID KS')
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.show()
else:
    print('scores_df not defined -- run Section F end-to-end first.')
